In [ ]:
'''# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session'''

### In this experiment, a 2-layer Graph Convolutional Network (GCN)  is implemented for node classification on the Amazon Computers dataset. GCN is a fundamental graph neural network model that aggregates information from a node’s local neighborhood to learn meaningful representations.Each layer in a GCN performs message passing by combining features from neighboring nodes, allowing the model to capture structural and feature-based information from the graph.

In [ ]:
!pip install torch-geometric


In [1]:
import torch
import torch_geometric

print(torch.__version__)
print(torch_geometric.__version__)

2.10.0+cu128
2.7.0


### Loading Amazon Computer Dataset

In [2]:
from torch_geometric.datasets import Amazon
dataset=Amazon(root='/kaggle/working/Amazon',name='Computers')
data=dataset[0]
print(data)

Processing...


Data(x=[13752, 767], edge_index=[2, 491722], y=[13752])


Done!


In [3]:
print(data.x.shape)
print(data.edge_index.shape)
print(data.y.shape)
print(data.y.unique())


torch.Size([13752, 767])
torch.Size([2, 491722])
torch.Size([13752])
tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])


### Doing Train Test Validation Split(70% train, 15% validation, 15% test)



In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
y=data.y.cpu().numpy()
indices=np.arange(len(y))
train_idx,temp_idx=train_test_split(indices,stratify=y,test_size=0.3)
val_idx,test_idx=train_test_split(temp_idx,stratify=y[temp_idx],test_size=0.5)
train_idx=torch.tensor(train_idx)
val_idx=torch.tensor(val_idx)
test_idx=torch.tensor(test_idx)




In [5]:
print(len(train_idx), len(val_idx), len(test_idx))

9626 2063 2063


In [6]:
#moving data to GPU
data = data.to(device)

### Mini-Batching with Neighbor Sampling

To efficiently train on large graphs, we use **NeighborLoader** for mini-batch training.

Instead of processing the entire graph at once, NeighborLoader samples a **subgraph** for each batch consisting of:

- **Target nodes**: Nodes for which predictions and loss are computed
- **Neighbor nodes**: Additional nodes included to provide context for message passing

### Loss Computation Strategy

Loss is computed **only on the target nodes** in each batch:

- The remaining nodes are included solely to support neighborhood aggregation

In [ ]:
import torch
print(torch.__version__)
print(torch.version.cuda)

In [ ]:
# Install dependencies from source 
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.10.0+cu128.html
!pip install torch-geometric

# IMPORTANT: Check if it actually installed correctly
try:
    import pyg_lib
    import torch_sparse
    print("Success: Libraries are ready!")
except ImportError:
    print("Binaries not found, trying an alternative install...")
    # If the above fails, you can try installing without the '-f' link to force a local build
    !pip install pyg-lib torch-scatter torch-sparse torch-cluster -U

In [7]:
import torch
import torch_sparse
import pyg_lib

print(f"PyG-Lib Version: {pyg_lib.__version__}")
print(f"Torch-Sparse Version: {torch_sparse.__version__}")

PyG-Lib Version: 0.6.0+pt210cu128
Torch-Sparse Version: 0.6.18+pt210cu128


In [8]:
from torch_geometric.loader import NeighborLoader

### Defining the 2-Layer GCN Architecture

In [9]:
from torch_geometric.nn import GCNConv
import torch.nn.functional as F
class GCN(torch.nn.Module):
    def __init__(self,in_channels,hidden_channels,out_channels):
        super().__init__()
        self.conv1=GCNConv(in_channels,hidden_channels)
        self.conv2=GCNConv(hidden_channels,out_channels)
    def forward(self,x,edge_index):
        x=self.conv1(x,edge_index)
        x=F.relu(x)
        x=self.conv2(x,edge_index)
        return x

### Training function to perform 1 epoch of training

In [10]:
def train(model,loader,optimizer,device):
    model.train()
    total_loss=0
    for batch in loader:
        batch=batch.to(device)
        optimizer.zero_grad()
        out=model(batch.x,batch.edge_index)
        loss=F.cross_entropy(out[:batch.batch_size],batch.y[:batch.batch_size])
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    return total_loss/len(loader)

### Validation function to evaluate the model on the validation set 

In [11]:
@torch.no_grad()
def evaluate(model,loader,device):
    model.eval()
    correct=0
    total=0
    for batch in loader:
        batch=batch.to(device)
        out=model(batch.x,batch.edge_index)
        pred=out[:batch.batch_size].argmax(dim=1)
        correct+=(pred==batch.y[:batch.batch_size]).sum().item()
        total+=batch.batch_size
    return correct/total

### Setting Random Seed

In [15]:
import random
import numpy as np
def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    

### Testing function to evaluate the final model on test set 

In [16]:
from sklearn.metrics import f1_score

@torch.no_grad()
def test_with_f1(model, loader, device):
    model.eval()
    
    all_preds = []
    all_labels = []

    for batch in loader:
        batch = batch.to(device)

        out = model(batch.x, batch.edge_index)
        pred = out[:batch.batch_size].argmax(dim=1)

        all_preds.append(pred.cpu())
        all_labels.append(batch.y[:batch.batch_size].cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    acc = (all_preds == all_labels).mean()
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    micro_f1 = f1_score(all_labels, all_preds, average='micro')

    return acc, macro_f1, micro_f1

In [17]:
#tuning hidden dimension and lr
hidden_dims = [16, 32, 64]
lrs = [0.01, 0.005, 0.001]

configs = []

for h in hidden_dims:
    for lr in lrs:
        config = {
            "hidden_dim": h,
            "lr": lr,
            "num_neighbors": [10, 10]
        }
        configs.append(config)

In [18]:
#tune no of neighbors
neighbor_options = [
    [5, 5],
    [10, 10],
    [15, 10],
    [20, 10]
]


### Experiment Pipeline

This function manages the end-to-end training and evaluation process.

- Initializes the model using the given configuration
- Constructs NeighborLoaders for mini-batch training
- Trains the model across epochs with early stopping based on validation performance
- Saves the best-performing model
- Evaluates final performance on the test set using Accuracy and F1 scores

In [19]:
def run_experiment(config, seed=0, run_test=False):
    set_seed(seed)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    #epochs=4
    epochs = 50 if not run_test else 100
    model = GCN(
        data.num_features,
        config["hidden_dim"],
        dataset.num_classes
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])


    train_loader = NeighborLoader(
        data,
        input_nodes=train_idx,
        num_neighbors=config["num_neighbors"],
        batch_size=128,
        shuffle=True
    )

    val_loader = NeighborLoader(
        data,
        input_nodes=val_idx,
        num_neighbors=config["num_neighbors"],
        batch_size=256,
        shuffle=False
    )

    best_val = 0
    patience = 10
    counter = 0
    save_model=run_test 

    for epoch in range(1,epochs+1):
        loss = train(model, train_loader, optimizer, device)
        val_acc = evaluate(model, val_loader, device)

        if val_acc > best_val:
            best_val = val_acc
            counter = 0
            if save_model:
             torch.save(model.state_dict(), f"/kaggle/working/best_modelGCN_{seed}.pt")
        else:
            counter += 1

        if counter >= patience:
            break
    if not run_test:
        return best_val

    # load best model
    model.load_state_dict(torch.load(f"/kaggle/working/best_modelGCN_{seed}.pt", map_location=device))

    if not run_test:
        return best_val

    test_loader = NeighborLoader(
        data,
        input_nodes=test_idx,
        num_neighbors=config["num_neighbors"],
        batch_size=256,
        shuffle=False
    )

    acc, macro_f1, micro_f1 = test_with_f1(model, test_loader, device)

    return acc, macro_f1,micro_f1
    

    

### Hyperparameter Tuning (for learning rate and hidden dimension)

In [20]:
#Hyperparameter Tuning (lr and hidden dimension)
best_val = 0
best_config = None

for config in configs:
    val_acc = run_experiment(config, seed=0, run_test=False)
    
    print(config, "→", val_acc)

    if val_acc > best_val:
        best_val = val_acc
        best_config = config

print("Best config:", best_config)

{'hidden_dim': 16, 'lr': 0.01, 'num_neighbors': [10, 10]} → 0.37518177411536596
{'hidden_dim': 16, 'lr': 0.005, 'num_neighbors': [10, 10]} → 0.37518177411536596
{'hidden_dim': 16, 'lr': 0.001, 'num_neighbors': [10, 10]} → 0.8773630634997577
{'hidden_dim': 32, 'lr': 0.01, 'num_neighbors': [10, 10]} → 0.8400387784779447
{'hidden_dim': 32, 'lr': 0.005, 'num_neighbors': [10, 10]} → 0.8763936015511391
{'hidden_dim': 32, 'lr': 0.001, 'num_neighbors': [10, 10]} → 0.8889966068831798
{'hidden_dim': 64, 'lr': 0.01, 'num_neighbors': [10, 10]} → 0.8759088705768299
{'hidden_dim': 64, 'lr': 0.005, 'num_neighbors': [10, 10]} → 0.8831798351914687
{'hidden_dim': 64, 'lr': 0.001, 'num_neighbors': [10, 10]} → 0.8841492971400873
Best config: {'hidden_dim': 32, 'lr': 0.001, 'num_neighbors': [10, 10]}


### Hyperparameter Tuning (Number of neighbors)

In [21]:
#Tune no of neighbors 
best_val = 0
best_neighbors = None

for neighbors in neighbor_options:
    
    config = {
        "hidden_dim": best_config["hidden_dim"],
        "lr": best_config["lr"],
        "num_neighbors": neighbors
    }

    val_acc = run_experiment(config, seed=0, run_test=False)

    print(f"{neighbors} → {val_acc:.4f}")

    if val_acc > best_val:
        best_val = val_acc
        best_neighbors = neighbors

print("Best neighbors:", best_neighbors)

[5, 5] → 0.8701
[10, 10] → 0.8788
[15, 10] → 0.8846
[20, 10] → 0.8958
Best neighbors: [20, 10]


In [22]:
best_config["num_neighbors"] = best_neighbors

### Final training  and evaluation of the model across 10 seed values

In [23]:
#Final Training
acc_list = []
macro_f1_list = []
micro_f1_list=[]
for seed in range(10):
    acc, macro_f1,micro_f1  = run_experiment(best_config, seed=seed, run_test=True)
    acc_list.append(acc)
    macro_f1_list.append(macro_f1)
    micro_f1_list.append(micro_f1)

    print(f"Seed {seed}: Acc={acc:.4f}, macro_F1={macro_f1:.4f},micro_F1={micro_f1:.4f}")

Seed 0: Acc=0.9045, macro_F1=0.9064,micro_F1=0.9045
Seed 1: Acc=0.8977, macro_F1=0.8910,micro_F1=0.8977
Seed 2: Acc=0.9021, macro_F1=0.9034,micro_F1=0.9021
Seed 3: Acc=0.8880, macro_F1=0.8786,micro_F1=0.8880
Seed 4: Acc=0.8924, macro_F1=0.8897,micro_F1=0.8924
Seed 5: Acc=0.9045, macro_F1=0.9051,micro_F1=0.9045
Seed 6: Acc=0.9079, macro_F1=0.9120,micro_F1=0.9079
Seed 7: Acc=0.9011, macro_F1=0.9035,micro_F1=0.9011
Seed 8: Acc=0.8972, macro_F1=0.8909,micro_F1=0.8972
Seed 9: Acc=0.8909, macro_F1=0.8899,micro_F1=0.8909


### The final performance is reported as mean and standard deviation of Accuracy and F1 scores across all runs


In [25]:
import numpy as np

acc_mean = np.mean(acc_list)
acc_std = np.std(acc_list)

macro_f1_mean = np.mean(macro_f1_list)
macro_f1_std = np.std(macro_f1_list)

micro_f1_mean = np.mean(micro_f1_list)
micro_f1_std = np.std(micro_f1_list)

print(f"\nFinal Results:")
print(f"Accuracy: {acc_mean:.4f} ± {acc_std:.4f}")
print(f"Macro F1: {macro_f1_mean:.4f} ± {macro_f1_std:.4f}")
print(f"Micro F1: {micro_f1_mean:.4f} ± {micro_f1_std:.4f}")


Final Results:
Accuracy: 0.8986 ± 0.0062
Macro F1: 0.8970 ± 0.0099
Micro F1: 0.8986 ± 0.0062
